#Preprocessing + TFIDF/LR Feature Importance exploartion

The main purpose of this notebook was to find different preprocessing methods to try to reduce data leakage in our model, and create a final preprocessing method to use across training for all of our models. A lot of different methods were used, but none had significant results. Methods and results not used were deleted to make the notebook more readable.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("zefang-liu/phishing-email-dataset")

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset['train'])
df.head()

,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df = df[["Email Text", "Email Type"]]
df.columns = ["text", "label"]
df.head()

,text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,the other side of * galicismos * * galicismo *...,Safe Email
2,re : equistar deal tickets are you still avail...,Safe Email
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [ ]:
df["label"] = df["label"].apply(lambda x: 1 if x == "Phishing Email" else 0)
df.head()

,text,label
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",0
1,the other side of * galicismos * * galicismo *...,0
2,re : equistar deal tickets are you still avail...,0
3,\nHello I am your hot lil horny toy.\n I am...,1
4,software at incredibly low prices ( 86 % lower...,1


#Cleaning to find duplicates and near duplicates
The code below standarizes text so the model does not memorize safe IDs, accounts, urls, etc. This method is used to find duplicates and near duplicates to detect any data leakage and remove repeating texts or avoid situations where our model memorizes dataset specific details during classification. To add on to cleaning, we filtered out text lengths greater than 500 and less than 20 because phishing emails had significantly shorter text lengths. This method mitigates our model from considering text lengths as a feature.

In [ ]:
import pandas as pd
import re
from collections import defaultdict

def clean_email_text(text):
    """Basic text cleaning for duplicate detection."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()      # normalize spaces
    return text

def further_clean(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"\S+@\S+", " <EMAIL> ", text)
    text = re.sub(r"\b\d+\b", " <NUM> ", text)
    text = re.sub(r"\b(?:jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\b", " <DATE> ", text)
    text = re.sub(r"\b(account|invoice|order|id|number)\s*\w*", r"\1 <ID>", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text
df["text"] = df["text"].apply(clean_email_text)
df["text"] = df["text"].apply(further_clean)

df.head()


,text,label
0,re <NUM> <NUM> disc uniformitarianism re <NUM>...,0
1,the other side of galicismos galicismo is a sp...,0
2,re equistar deal tickets are you still availab...,0
3,hello i am your hot lil horny toy i am the one...,1
4,software at incredibly low prices <NUM> lower ...,1


In [ ]:
df = df.drop_duplicates()
df = df.dropna()
print(len(df))

17136


In [ ]:
df["text_length"] = df["text"].str.split().str.len()

df.groupby("label")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,10691.0,547.966327,24214.441023,0.0,76.5,163.0,324.5,2502955.0
1,6445.0,263.149263,495.822992,0.0,60.0,117.0,256.0,11751.0


In [ ]:
df = df[df["text_length"] <= 500]
df = df[df["text_length"] > 20]

df.groupby("label")["text_length"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
0,8741.0,172.383137,119.145083,21.0,75.0,143.0,245.00,500.0
1,5306.0,142.676027,110.229270,21.0,62.0,107.0,186.75,500.0


In [ ]:
len(df)

14047

#Explanation
Our preprocessing methods were super aggressive and got rid of 400 rows of data that contributed to duplicates/near duplicates, empty values, or noisy data. We decided to stop at removing any more rows of data from our dataset because this would lead to hurting the dataset more than cleaning.

#80/10/10 split
create dataframes for trainng, validation, and testing

In [ ]:
from sklearn.model_selection import train_test_split

x = df["text"]
y = df["label"]

# split into train(80) and temp (20)
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y,
    random_state=3,
    test_size=0.2,
    stratify=y)

print(x_train.shape)
print(x_temp.shape)

(11237,)
(2810,)


In [ ]:
x_valid, x_test, y_valid, y_test = train_test_split(
    x_temp, y_temp,
    random_state=3,
    test_size=0.5,
    stratify=y_temp)

print(x_valid.shape)
print(x_test.shape)

(1405,)
(1405,)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# capturing unigrams & bigrams
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2))

x_train_tfidf = tfidf.fit_transform(x_train)
x_valid_tfidf = tfidf.transform(x_valid)

In [ ]:
feature_names = tfidf.get_feature_names_out()
print(feature_names[:20])

['1st' '2nd' '3d' '3d 3d' '3rd' 'aa' 'ab' 'ability' 'ability to' 'able'
 'able to' 'about' 'about all' 'about how' 'about it' 'about num'
 'about our' 'about the' 'about this' 'about to']


In [ ]:
coefficients = classifier.coef_[0]

In [ ]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

# most positive coefficients
top_positive = feature_importance.sort_values("coefficient", ascending=False).head(20)

# most negative coefficients
top_negative = feature_importance.sort_values("coefficient", ascending=True).head(20)

print("Top phishing email features:")
print(top_positive)

print("\nTop safe email features:")
print(top_negative)

Top phishing email features:
         feature  coefficient
9903        your     5.840430
9805         you     4.941171
3505        here     4.182527
3649        http     4.098833
6021         our     4.079967
5326          no     3.340959
1504       click     3.009878
1506  click here     2.922327
5088       money     2.848024
6912      remove     2.676535
3043        free     2.623950
4503        life     2.444196
9276      viagra     2.424891
3907        info     2.415080
7551    software     2.381202
7484        site     2.293613
4689         low     2.254677
7185        save     2.239728
5893      online     2.192181
6076    over num     2.125845

Top safe email features:
          feature  coefficient
2501        enron    -6.216857
9755        wrote    -3.797446
8103          the    -3.709178
9135          url    -3.641455
5820           on    -3.602498
9136     url date    -3.559723
8047         that    -3.501657
8038       thanks    -3.414653
1946     date num    -2.999185
1937 

#Feature Extraction

Here is some basic exploration of the unigram and bigrams that contributed most towards the models predicitons. The coefficients next to the grams show how heavily each word/phrase was weighed. There are very clear patterns in the contribution toward probability outcome.

In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression(max_iter=1000, random_state=42)
classifier.fit(x_train_tfidf, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [ ]:
y_pred = classifier.predict(x_valid_tfidf)
y_prob = classifier.predict_proba(x_valid_tfidf)[:, 1]

# evaluate
from sklearn.metrics import classification_report

print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.99      0.98       874
           1       0.98      0.96      0.97       531

    accuracy                           0.98      1405
   macro avg       0.98      0.97      0.97      1405
weighted avg       0.98      0.98      0.98      1405



#Evaluating preprocessing

I evaluated the preprocessed data on a simple architecture because it was easy and quick to train and test. Everytime I ran the model, the accuracy results are still super high. I do not think the model is achieveing high accuracy due to data leakage because I used jaccard similarity thresholds on trigram overlap to remove near duplicates, and there was still a high accuracy no matter how many rows I removed. The data must be separable by other factors. I landed on this set of methods to show an aggressive but fair method for cleaning the text data for emails.